<a href="https://colab.research.google.com/github/Val095/InteligenciaArtificial_II/blob/main/SVM_Housing_USA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files  # Importa la herramienta de Colab para cargar el archivo desde el computador.
import io  # Importa utilidades para leer los bytes del archivo cargado en memoria.
import pandas as pd  # Importa pandas para manipular el dataset en tablas.
from sklearn.model_selection import train_test_split  # Importa la función para separar datos de entrenamiento y prueba.
from sklearn.preprocessing import StandardScaler  # Importa el escalador necesario para normalizar las variables numéricas.
from sklearn.svm import SVR  # Importa Support Vector Regression para construir los modelos SVM de regresión.
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Importa las métricas para evaluar las predicciones.
cargado = files.upload()  # Abre el selector de archivos de Colab para cargar Housing.csv desde el equipo.
df = pd.read_csv(io.BytesIO(next(iter(cargado.values()))))  # Lee el CSV cargado y lo convierte en un DataFrame de pandas.
display(df.head())  # Muestra las primeras filas para conocer la estructura inicial de las viviendas de Estados Unidos.
print("Columnas disponibles:", df.columns.tolist())  # Imprime las columnas para verificar las variables existentes en el dataset.
candidatas = [c for c in df.columns if c.lower() in {"price", "value", "precio", "valor"}]  # Busca una columna que represente el precio o valor de la vivienda.
if not candidatas: raise ValueError("No se encontró una columna de precio o valor.")  # Detiene el proceso si no existe una variable objetivo válida.
objetivo = candidatas[0]  # Selecciona la columna real encontrada, que en este dataset es price, como variable objetivo.
X = pd.get_dummies(df.drop(columns=objetivo), drop_first=True)  # Crea las predictoras y convierte categorías como yes/no en variables numéricas.
y = df[objetivo]  # Guarda el precio de la vivienda como la variable que se desea predecir.
print(f"Objetivo: {objetivo}\nPredictoras: {X.columns.tolist()}")  # Muestra la variable objetivo y las predictoras finales usadas por el modelo.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # Separa 80% para entrenar y 20% para evaluar de forma reproducible.
escalador = StandardScaler()  # Crea el objeto que estandariza las predictoras para que tengan escalas comparables.
X_train_esc = escalador.fit_transform(X_train)  # Calcula la escala usando entrenamiento y transforma sus variables.
X_test_esc = escalador.transform(X_test)  # Transforma prueba con la escala aprendida para evitar filtrar información del conjunto de prueba.
svr_lineal = SVR(kernel="linear")  # Crea y entrena un SVR con relaciones lineales entre características y precio.
svr_lineal.fit(X_train_esc, y_train)  # Ajusta el modelo lineal usando las viviendas de entrenamiento.
svr_polinomial = SVR(kernel="poly", degree=3)  # Crea y entrena un SVR polinomial de grado tres para capturar relaciones no lineales.
svr_polinomial.fit(X_train_esc, y_train)  # Ajusta el modelo polinomial usando las viviendas de entrenamiento.
pred_lineal = svr_lineal.predict(X_test_esc)  # Predice los precios del conjunto de prueba con el SVR lineal.
pred_polinomial = svr_polinomial.predict(X_test_esc)  # Predice los precios del conjunto de prueba con el SVR polinomial.
metricas = lambda pred: [mean_absolute_error(y_test, pred), mean_squared_error(y_test, pred)**0.5, r2_score(y_test, pred)]  # Calcula MAE, RMSE y R² para unas predicciones.
resultados = pd.DataFrame([["SVR lineal", *metricas(pred_lineal)], ["SVR polinomial", *metricas(pred_polinomial)]], columns=["Modelo", "MAE", "RMSE", "R²"])  # Construye una tabla comparativa de ambos modelos.
display(resultados.round(2))  # Muestra las métricas redondeadas para facilitar la comparación de los modelos.

Saving Housing.csv to Housing.csv


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


Columnas disponibles: ['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']
Objetivo: price
Predictoras: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'mainroad_yes', 'guestroom_yes', 'basement_yes', 'hotwaterheating_yes', 'airconditioning_yes', 'prefarea_yes', 'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished']


,Modelo,MAE,RMSE,R²
0,SVR lineal,1763405.24,2359167.90,-0.1
1,SVR polinomial,1763889.61,2359641.57,-0.1
